In [1]:
%pip install -r ../../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "tabularisai/Qwen3-0.3B-distil"
local_model_path = "./pretrained-qwen3-0.3B"

synthetizer_path="qwen-distil.pickle"
synth_data_output_path="qwen_synth_out.csv"

print(f"Descargando {model_name}...")

# Descargamos Tokenizer y Modelo
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Guardamos en local
tokenizer.save_pretrained(local_model_path)
model.save_pretrained(local_model_path)

print(f"¡Listo! Copia la carpeta '{local_model_path}' a tu máquina sin internet.")

/home/aperez/workspace/synthetic-data-learning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Descargando tabularisai/Qwen3-0.3B-distil...


Writing model shards: 100%|██████████| 1/1 [00:06<00:00,  6.46s/it]

¡Listo! Copia la carpeta './pretrained-qwen3-0.3B' a tu máquina sin internet.


In [3]:
import pandas as pd
import numpy as np

from be_great import GReaT
from sdv.metadata import Metadata
from sdv.evaluation import evaluate_quality

# 1. Cargar el dataset adjunto
df_real = pd.read_csv('../../datasources/bank_full_dataset/bank-full.csv', sep=';')

# Para la PoC, tomamos una muestra representativa para optimizar tiempos de GPU/CPU
# En producción (Vertex AI), usaríamos el dataset completo.
df_train = df_real.sample(n=1000, random_state=42)
metadata = Metadata.detect_from_dataframes(
    data={'bank_full': df_real}
) 

print(f"Dataset cargado: {df_real.shape[0]} registros.")
print("Columnas detectadas:", df_real.columns.tolist())

num_cols = df_real.select_dtypes(include=[np.number]).columns
print("Columnas numericas:", num_cols)
cat_cols = df_real.select_dtypes(exclude=[np.number]).columns
print("Columnas categoricas:", cat_cols)

Dataset cargado: 45211 registros.
Columnas detectadas: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']
Columnas numericas: Index(['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous'], dtype='object')
Columnas categoricas: Index(['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact',
       'month', 'poutcome', 'y'],
      dtype='object')


In [4]:
import pandas as pd
import torch
from be_great import GReaT

# 1. Ruta donde copiaste la carpeta

# 3. Inicializamos GReaT apuntando a la ruta local
# Importante: llm=local_model_path evita que intente conectar a internet
model = GReaT(
    llm=local_model_path,  
    float_precision=2,  # Limit floating-point precision to 3 decimal places
    batch_size=16,     # Use smaller batch size for small datasets
    epochs=16,          # Train for more epochs with small data
    bf16=torch.cuda.is_available()  # bf16 solo con GPU: transformers falla si se activa en CPU
)

# Entrenamiento
model.fit(df_train)
model.save(synthetizer_path)


Loading weights: 100%|██████████| 156/156 [00:00<00:00, 2381.50it/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 151643}.


{'loss': '0.9162', 'grad_norm': '3.922', 'learning_rate': '2.525e-05', 'epoch': '7.937'}


Writing model shards: 100%|██████████| 1/1 [00:10<00:00, 10.54s/it]


{'loss': '0.8212', 'grad_norm': '4.25', 'learning_rate': '4.464e-07', 'epoch': '15.87'}


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.96s/it]


{'train_runtime': '508.7', 'train_samples_per_second': '31.45', 'train_steps_per_second': '1.981', 'train_loss': '0.8683', 'epoch': '16'}


/home/aperez/workspace/synthetic-data-learning/.venv/lib/python3.12/site-packages/be_great/great.py:1209: UserWarning: Directory qwen-distil.pickle already exists and is overwritten now.
  warnings.warn(f"Directory {path} already exists and is overwritten now.")


In [5]:
# 3. Generación de datos sintéticos
# Vamos a generar 1000 registros nuevos "soñados" por el modelo
df_synthetic = model.sample(n_samples=1000, guided_sampling=True)

# Limpieza rápida: GREAT a veces genera strings con espacios extra
df_synthetic = df_synthetic.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

print("Muestra de datos sintéticos generados:")
display(df_synthetic.head())

df_synthetic.to_csv(synth_data_output_path)

Muestra de datos sintéticos generados:


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,40,blue-collar,married,secondary,no,239,yes,no,unknown,20,may,484,3,-1,0,unknown,no
1,39,technician,single,secondary,no,56,yes,no,unknown,18,may,325,1,-1,0,unknown,no
2,32,management,married,tertiary,no,1258,no,no,cellular,3,aug,169,2,-1,0,unknown,no
3,28,management,single,tertiary,no,359,yes,no,unknown,12,jun,1061,2,-1,0,unknown,no
4,33,blue-collar,single,secondary,no,-253,yes,no,cellular,28,nov,192,1,165,2,failure,no


In [6]:
# 5. Función de reporte de calidad
def reporte_calidad(real, synth):
    print("\n" + "="*40)
    print("   REPORTE DE FIDELIDAD ESTADÍSTICA")
    print("="*40 + "\n")
      
    # Test de Proporciones para Categóricas (Diferencia Media)
    cat_cols = real.select_dtypes(include=['object', 'category']).columns
    for col in cat_cols:
        f_real = real[col].value_counts(normalize=True).sort_index()
        f_synth = synth[col].value_counts(normalize=True).sort_index()
        diff = (f_real - f_synth).abs().mean()
        status = "✅ FIEL" if diff < 0.05 else "❌ DESVIADO"
        print(f"Prop. Diff [{col:10}]: {diff:.4f} {status}")

reporte_calidad(df_real, df_synthetic)


   REPORTE DE FIDELIDAD ESTADÍSTICA

Prop. Diff [job       ]: 0.0225 ✅ FIEL
Prop. Diff [marital   ]: 0.0279 ✅ FIEL
Prop. Diff [education ]: 0.0397 ✅ FIEL
Prop. Diff [default   ]: 0.0170 ✅ FIEL
Prop. Diff [housing   ]: 0.0178 ✅ FIEL
Prop. Diff [loan      ]: 0.1022 ❌ DESVIADO
Prop. Diff [contact   ]: 0.0993 ❌ DESVIADO
Prop. Diff [month     ]: 0.0236 ✅ FIEL
Prop. Diff [poutcome  ]: 0.0538 ❌ DESVIADO
Prop. Diff [y         ]: 0.0630 ❌ DESVIADO


In [7]:
import torch
import gc

# 1. Eliminar el objeto del modelo (y cualquier otro pesado)
if 'model' in locals():
    del model
if 'df_synthetic' in locals():
    del df_synthetic

# 2. Forzar la recolección de basura de Python
gc.collect()

# 3. Vaciar el caché de PyTorch (esto libera la VRAM físicamente)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    
print("Memoria de GPU liberada.")

Memoria de GPU liberada.
